In [0]:
#silver cleaning
from pyspark.sql import functions as F

df = spark.read.table("dbr_dev_ua5816bd.team_tristar_bronze.calendar_dates")

df = df.withColumn(
    'service_id',
    F.when(
        F.col('service_id').try_cast('int').isNull(),
        None
    ).otherwise(
        F.col('service_id').try_cast('int')
    )
)

df = df.withColumn(
    'date',
    F.when(
        F.col('date').try_cast('int').isNull(),
        None
    ).otherwise(
        F.col('date').try_cast('int')
    )
)

df = df.withColumn(
    'exception_type',
    F.when(
        ~F.col('exception_type').try_cast('int').isin(1, 2),
        None
    ).otherwise(
        F.col('exception_type').try_cast('int')
    )
)

df = df.withColumn(
    'source',
    F.when(
        F.trim(F.col('source')) == '',
        None
    ).otherwise(
        F.trim(F.col('source'))
    )
)

df = df.withColumn(
    'source_update_date',
    F.when(
        F.trim(F.col('source_update_date')) == '',
        None
    ).otherwise(
        F.trim(F.col('source_update_date'))
    )
)

In [0]:
#merge
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "dbr_dev_ua5816bd.team_tristar_silver.calendar_dates"
)

silver_table.alias("silver").merge(
    df.alias("bronze"),
    "silver.service_id = bronze.service_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()